# Client Boto3 / S3

In [1]:
import os
import io 
import boto3
import json
from pprint import pprint
import pandas as pd
from tqdm.notebook import tqdm  #  Barre Jupyter native (bleue)
# ou : from tqdm.autonotebook import tqdm  # auto console/notebook

import sys
import statistics

import html
from bs4 import BeautifulSoup
import re
from typing import Dict, List, Optional, Any

import pyarrow.parquet as pq
import pyarrow.json as paj
import pyarrow as pa

from tqdm import tqdm
import time

endpoint = os.environ["S3_ENDPOINT_URL"]
bucket = os.environ["S3_BUCKET"]

s3_boto = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)

class WTTJ:
    def __init__(self, job_title: str, job_description: int):
        self.job_title = job_title
        self.job_description = job_description




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

# Class 

In [2]:
# ==========================
# Class WTTJ
# ==========================

class WTTJ:
    def __init__(self, id :int,  job_title: str, job_description: str, company_employees : int, profile : str, url : str):
        self.id = id       
        self.job_title = job_title
        self.job_description = job_description
        self.company_employees= company_employees
        self.profile = profile
        self.url = url

    def __str__(self) -> str:
        """Surcharge affichage print()"""
        desc = self.job_description[:50] if self.job_description else "NA"
        profile = self.profile[:50] if self.profile else "NA"
        
        return f"""- Id : #{self.id}
        job_title : {self.job_title}
        job_description : {desc}...
        company_employees : {self.company_employees}
        profile : {profile}
        url : {self.url}"""


# Helper

In [3]:
def get_json_field_from_record(record, field_name):
    if isinstance(record[field_name], str):
        data = json.loads(record[field_name])
    else:
        data = record[field_name]  # déjà un dict !
    return data

def get_prefix_keys(s3_client, prefix):
    keys = []
    
    # Paginator + tqdm
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    pbar = tqdm(total=0, desc="Listing S3 keys", unit="keys")
    
    for page in page_iterator:
        page_keys = [obj["Key"] for obj in page.get("Contents", [])]
        keys.extend(page_keys)
        pbar.update(len(page_keys))
    
    pbar.close()
    return keys

def get_json_lines_from_jsonld( s3_client, key):
    obj = s3_client.get_object(Bucket=bucket, Key = key)
    lines = obj["Body"].read().decode("utf-8").splitlines()
    return lines

def clean_html(data):
    if isinstance(data, str):
        # Remove html tag
        soup = BeautifulSoup(data, 'html.parser')
        text = soup.get_text()
        # Remove Html Entities
        return html.unescape(text).strip()
    return data

def human_readable(size_bytes):
    for unit in ['B', 'Ko', 'Mo', 'Go', 'To']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.1f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.1f} Po"

def get_stats_from_list(all_lines) :
    # Taille moyenne payload (octets)
    sizes = [sys.getsizeof(line) for line in all_lines]
    taille_moyenne_octets = statistics.mean(sizes)
    taille_moyenne_mo = taille_moyenne_octets / (1024*1024)
    
    print(f"- Nb jsonld: {len(all_lines)}")
    print(f"- Taille moyenne: {human_readable(statistics.mean(sizes))}")
    print(f"- Taille totale: {human_readable(sum(sizes))}")

def find_field_in_json(data, target_field, path=[]):
    """
    Cherche champ récursivement.
    
    >>> find_field_in_json(record, 'name')
    [{'path': ['job_data', 'name'], 'value': 'Stagiaire...'}]
    """
    matches = []
    
    if isinstance(data, dict):
        if target_field in data:
            matches.append({
                'path': path + [target_field],
                'value': data[target_field]
            })
        
        for key, value in data.items():
            matches.extend(find_field_in_json(value, target_field, path + [key]))
    
    elif isinstance(data, list):
        for i, item in enumerate(data):
            matches.extend(find_field_in_json(item, target_field, path + [f"[{i}]"]))
    
    return matches

def get_first_field(data, field_name):
    """Premier match."""
    matches = find_field_in_json(data, field_name)
    return matches[0]['value'] if matches else None


def get_field_or_default(data, field_name, default=None):
    """Premier match, préserve type."""
    matches = find_field_in_json(data, field_name)
    if matches:
        value = matches[0]['value']
        # Préserve listes/tableaux
        if isinstance(value, (list, dict)):
            return value
        return str(value)[:1000]  # Tronque strings longs
    return default
# ================
# Fixe double encoding issue on json by recursive parsing Apollo''s json
# ================

def _fix_double_encoding(text: str) -> str:
    """
    Fix double UTF-8 encoding issues.
    Example: 'M\u00c3\u00a9canique' -> 'Mécanique'
    """
    try:
        # 1. Decode HTML entities (&#39; -> ', &lt; -> <, etc.)
        text = html.unescape(text)
         
        # 2. Encode as latin-1 to get original UTF-8 bytes, then decode as UTF-8
        return text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        # If it fails, return original text
        return text


def _fix_double_encoded_dict(obj: Any) -> Any:
    """Recursively fix double UTF-8 encoding in dict/list structures."""
    if isinstance(obj, dict):
        return {k: _fix_double_encoded_dict(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_fix_double_encoded_dict(item) for item in obj]
    elif isinstance(obj, str):
        return _fix_double_encoding(obj)
    else:
        return obj
# ================

# ================
# Populate object with json
# ================

def set_wttj_all_from_json(jsonld_keys, s3_boto):
    dfs = []
    total_jobs = 0
    
    # BARRE PRINCIPALE UNIQUEMENT (position=0)
    with tqdm(
        total=len(jsonld_keys),
        desc="ETL WTTJ",
        position=0,
        leave=True,
        dynamic_ncols=True,  # ← Auto-width
        mininterval=0.1,
        smoothing=0.1
    ) as pbar:
        
        for i, key in enumerate(jsonld_keys):
            # FONCTION SANS tqdm interne !
            df, stats = set_wttj_all_from_json_silent(s3_boto, key)
            dfs.append(df)
            total_jobs += stats["added"]
            
            # Postfix COURT
            postfix = (
                f"J:{total_jobs:,} | "
                f"+{stats['added']:,} | "
                f"{i+1}/{len(jsonld_keys)}"
            )
            
            pbar.set_postfix_str(postfix, refresh=True)
            pbar.update(1)
    
    return pd.concat(dfs, ignore_index=True)


def set_wttj_all_from_json_silent(s3_client, key):
    """AUCUN print/tqdm → SILENT."""
    data = []
    errors = 0
    
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        # Iterlines to preserve memory load
        for line_bytes in obj["Body"].iter_lines():
            #print(line_bytes[:3000])
            line = line_bytes.decode('utf-8', errors='ignore').strip()
           
            if not line: continue
            
            try:

                # 1. Parser le JSON
                record = json.loads(line)
                #print(f"🔍 Record keys: {list(record.keys())[:5]}")  # DEBUG
                
                # 2. Corriger le double encodage
                record = _fix_double_encoded_dict(record)
                
                job_data = get_json_field_from_record(record, "job_data")
                #print(f"🔍 job_data type: {type(job_data)}, keys: {list(job_data.keys())[:5] if isinstance(job_data, dict) else 'N/A'}")  # DEBUG
                
                initial_data = get_json_field_from_record(record, "initial_data")
                
                # Vérifier que job_data existe
                if not job_data or not isinstance(job_data, dict):
                    print(f"⚠️ job_data invalide à la ligne {lines_read}")
                    errors += 1
                    continue
                
                #pprint(initial_data)
                urls_list = job_data.get("urls", [])
                canonical_url = next(
                    (link.get('href', '') for link in urls_list if link.get('kind') == 'canonical'),
                    ''
                )
               
                data.append({
                    "wttj_reference": job_data.get("wttj_reference"),
                    "reference": job_data.get("reference"),                    
                    "name": clean_html(job_data.get("name", "")),
                    "description": clean_html(job_data.get("description", "")),
                    "profile": clean_html(job_data.get("profile")),

                    "salary_min": job_data.get("salary_min"),
                    "salary_max": job_data.get("salary_max"),
                    "salary_currency": job_data.get("salary_currency"),                    
                    "education_level": job_data.get("education_level"),
                    "company_summary": job_data.get("company_summary"),
                    "company_description": job_data.get("company_description"),

                    "updated_at": job_data.get("updated_at"),
                    "published_at": job_data.get("published_at"),
                    "archived_at": job_data.get("archived_at"),
                    
                    "contract_duration_min": job_data.get("contract_duration_min"),                    
                    "remote": job_data.get("remote"),
                    "ats": job_data.get("ats"),
                    "contract_duration_max": job_data.get("contract_duration_max"),
                    "experience_level": job_data.get("experience_level"),
                    "contract_type": job_data.get("contract_type"),
                    
                    "urls": urls_list,  
                    "canonical_url" : canonical_url,
                    "skills": job_data.get("skills", [""]),                    
                    "key_missions": job_data.get("key_missions", [""]),
                    "offices": job_data.get("offices", [""]),

                    "sectors" : get_field_or_default(record, 'sectors', []),
                    "profession" : get_field_or_default(record, 'profession'),
                })
            except:
                errors += 1
                print(f"⚠️ Error parsing line {lines_read} in {key}: {e}")
                # DEBUG: afficher la ligne problématique
                if errors <= 3:  # Limite à 3 exemples
                    print(f"   Line: {line[:200]}...")
                        
        
        df = pd.DataFrame(data)
        return df, {"added": len(df), "errors": errors}
    
    except:
        return pd.DataFrame(), {"added": 0, "errors": 1}


# Lister les objets d'un chemin

In [6]:
prefix = "bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw"
keys = get_prefix_keys(s3_boto,prefix)
print(f"- {len( keys)} elements in  {prefix} " )
print ( "First elements : " , keys[:5])



Listing S3 keys: 0keys [00:00, ?keys/s]
Listing S3 keys: 87keys [00:00, 276.67keys/s]

- 87 elements in  bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw 
First elements :  ['bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000001.jsonl', 'bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000002.jsonl', 'bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000003.jsonl', 'bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000004.jsonl', 'bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000005.jsonl']


# Liste des WTTJ Json

In [7]:
if False:
    prefix = "bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw"
    #prefix = "welcometothejungle/bronze/dt=2026-02-20/run_id=20260220T081356Z/segment=jobs_raw"
    jsonld_keys = get_prefix_keys(s3_boto, prefix)
    
    all_lines = []
    counter_lines=0
    pbar = tqdm(jsonld_keys, desc="Processing jsonld files")
    
    # Chargement de toutes les json => Ca peut exploser en mémoire
    # on drop la variable à la fin 
    for jsonld_key in pbar :
        lines = get_json_lines_from_jsonld( s3_boto , jsonld_key )
        counter_lines += len(lines)
        pbar.set_description(f"Processing {jsonld_key} - # lines : {len(lines)} ")
        all_lines.extend(lines)
    
    print(10* "=")
    print("Statistics")
    print(10* "=")
    get_stats_from_list(all_lines)
    
    #print("")
    #print("First element")
    #pprint(all_lines[0])
    
    del all_lines


# All WTTJ Object in DataFrame

In [8]:
prefix = "bronze/welcometothejungle/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw"
#prefix = "welcometothejungle/bronze/dt=2026-02-20/run_id=20260220T081356Z/segment=jobs_raw"

# Get Json ld keys
jsonld_keys = get_prefix_keys(s3_boto, prefix)

# Limite du dataset au chargement des x premiers jsonl =>Dataset de : x * nbLine dans le json 
#jsonld_keys = jsonld_keys[:10]

# Fix manualy json ld keys
#jsonld_keys = ['welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000001.jsonl']
#jsonld_keys = ['welcometothejungle/bronze/dt=2026-02-20/run_id=20260220T081356Z/segment=jobs_raw/part-000001.jsonl']

# Lancement
wttj_df = set_wttj_all_from_json(jsonld_keys, s3_boto)
#print(f"💾 {len(wttj_df):,} jobs → Parquet !")
#wttj_df.to_parquet("wttj_final.parquet")



Listing S3 keys: 0keys [00:00, ?keys/s]
Listing S3 keys: 87keys [00:00, 350.51keys/s]
ETL WTTJ: 100%|██████████| 87/87 [07:07<00:00,  4.92s/it, J:60,502 | +0 | 87/87]    


In [23]:
pd.set_option('display.max_colwidth', None)
#display(wttj_df.drop('description', axis=1, inplace=True))

display(wttj_df.head(1))


,wttj_reference,reference,name,description,profile,salary_min,salary_max,salary_currency,education_level,company_summary,...,contract_duration_max,experience_level,contract_type,urls,canonical_url,skills,key_missions,offices,sectors,profession
0,4fd1fc3b-5d30-4711-91f5-0eb6b3786628,ACADO_ll91OX2,Professeur particulier pour tous les niveaux en physique-chimie à Brétigny-sur-Orge (91220) - H/F,"Nous recherchons actuellement 5 enseignant(e)s à Brétigny-sur-Orge, pour accompagner nos élèves, et en particulier : Niveau : première Matière : physique-chimie Objectifs : gagner en régularité dans ses révisions et anticiper les exigences du supérieur en physique-chimie Rythme : 1h par semaine Disponibilité : sous 48h Rémunération : entre 18€ et 39€ brut par heure (selon profil) Expérience : idéalement 1 an minimum d'enseignement en soutien scolaire Type de contrat : CDD (temps plein, temps partiel) Acadomia cherche des enseignants à Brétigny-sur-Orge et ses alentours Vous souhaitez intervenir dans une autre discipline ou pour un autre niveau ? Pas d'inquiétude, nous recherchons des enseignants dans toutes les matières et toutes les classes pour dispenser des cours à Brétigny-sur-Orge (Carouge Joncs Marins, Cendrennes Babin, Moinerie Maison Neuve, Vétille Quatre-Vingts Arpents, La Fontaine Daumones, etc.) ainsi que dans les villes environnantes (Sainte-Geneviève-des-Bois, Saint-Michel-sur-Orge, Arpajon) : n'hésitez pas à nous faire parvenir votre candidature ! Votre profil pour enseigner Afin d'accompagner nos élèves dans toutes les disciplines, nous sommes à la recherche de pédagogues ayant au minimum un diplôme bac+3 validé, de tout profil : étudiants en prépa (MP, PSI, PC) ou ayant un bac+3 en poche, voulant un job étudiant axé sur l'enseignement de la physique-chimie Professeurs ou ex-enseignants souhaitant diversifier leurs pratiques Retraités motivés pour partager leurs savoirs Personnes désireuses de faire aimer et connaître leur passion pour la physique et/ou la chimie ainsi que leurs compétences dans ces disciplines Votre rémunération en tant qu'enseignant Acadomia Votre rémunération pourra être variable. Elle sera déterminée en fonction de plusieurs critères comme la classe de l'élève, la durée des sessions et votre expérience. Votre rémunération sera aussi tributaire du nombre d'élèves suivis : nos enseignants accompagnent en moyenne 4 élèves par semaine. Vos avantages de prof particulier Acadomia Des élèves assurés, dans les secteurs géographiques que vous choisissez Une expérience valorisante pour votre CV Une rémunération fixe à chaque fin de mois, fonction du nombre d'élèves que vous souhaitez Toute la gestion administrative prise en charge par Acadomia (charges, cotisation retraite, etc.) Des ressources pédagogiques à disposition et une application mobile pour gérer vos cours La possibilité de donner des cours collectifs ou des stages de vacances en centre, ou même en ligne Des horaires adaptables, compatibles avec un autre emploi Vos compétences clés comme professeur particulier de physique-chimie Bonne maîtrise des contenus scolaires selon les niveaux de physique-chimie enseignés (structure de la matière, radioactivité, chimie organique...) Approche pédagogique personnalisée et bienveillante Aptitude à montrer comment aborder les phénomènes à différentes échelles (du microscopique au macroscopique) Votre parcours candidat Un entretien téléphonique pour cerner votre parcours, vos objectifs et votre personnalité. Un échange pour évaluer vos capacités pédagogiques, techniques et relationnelles pour l'enseignement de la physique-chimie. Une rencontre avec votre référent enseignant dans votre centre Acadomia de rattachement. En moins d'une semaine, vous pouvez commencer à enseigner à vos premiers élèves, si vous nous rejoignez. à propos d'Acadomia Leader du soutien scolaire en France (110 centres, 100 000 familles accompagnées), nous proposons des solutions personnalisées de soutien scolaire et d'apprentissage à domicile, en

# Check sur une url d'annonce

In [ ]:
pd.set_option('display.max_colwidth', None)
print(wttj_df.describe())
print(wttj_df.columns)


# Calculé lors de la création du df
#wttj_df['canonical_url'] = wttj_df['urls'].apply(
#    lambda urls: next(
#        (link.get('href', '') for link in urls if link.get('kind') == 'canonical'),
#        ''
#    )
#)

#test_df = wttj_df.copy()
#test_df['urls'] = test_df['urls'].apply(
#    lambda urls: next(
#        (link.get('href', '') for link in urls if link.get('kind') == 'canonical'),
#        ''  # Default vide
#    )
#)

test_df = wttj_df[
    wttj_df['wttj_reference'].str.contains('d3553b48-59d0-4bd7-b8b8-17cb7f1db8e9', na=False)
]

display( test_df.head(3) ) 

# Affichage unitaire json

In [3]:
key = "welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000047.jsonl"

obj = s3_boto.get_object(Bucket=bucket, Key=key)
lines = obj["Body"].read().decode("utf-8").splitlines()
records = [json.loads(l) for l in lines if l.strip()]

print("- Dict. Keys = ", records[0].keys())
job_data = get_json_field_from_record(records[0], "job_data")

# 1. Taille réelle
response = s3_boto.head_object(Bucket=bucket, Key=key)
print(f"Taille S3: {response['ContentLength']:,} bytes")

# 2. Test download complet
obj = s3_boto.get_object(Bucket=bucket, Key=key)
content = obj["Body"].read()
print(f"Download: {len(content):,} bytes → OK ? {len(content) == response['ContentLength']}")

# 3. Lignes finales
lines = content.decode('utf-8', errors='ignore').splitlines()
print("Dernière ligne:", repr(lines[-1][:100]))
print("Nombre lignes:", len([l for l in lines if l.strip()]))

pprint(job_data)


- Dict. Keys =  dict_keys(['source', 'segment', 'url', 'fetched_at', 'status_code', 'ok', 'error', 'key', 'initial_data', 'job_data', 'parser_version'])
Taille S3: 128,953,522 bytes
Download: 128,953,522 bytes → OK ? True
Dernière ligne: '{"source": "welcometothejungle", "segment": "jobs", "url": "https://www.welcometothejungle.com/fr/co'
Nombre lignes: 1500
{'application_fields': [{'id': '7cbaac98-16dd-4311-84d9-b0154e7ff6db',
                         'mode': 'optional',
                         'name': 'cover_letter'},
                        {'id': '4b5d7fb1-7603-48ef-a29e-5c42b752523f',
                         'mode': 'mandatory',
                         'name': 'resume'},
                        {'id': 'd71f0a73-c0a5-4083-a77d-b7f9aeb8bf1d',
                         'mode': 'mandatory',
                         'name': 'portfolio'},
                        {'id': '4de51a8d-0902-4d76-9fbd-a80af4260033',
                         'mode': 'optional',
                         'name':

# Test Json

In [ ]:
def test_jsonl_loading_FULL(s3_client, key, bucket, max_lines=None):  # None = TOUT
    """Test COMPLET (toutes lignes)."""
    print(f"\n🧪 FULL {key.split('/')[-1]}")
    
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    total_lines = 0
    valid_json = 0
    job_data_ok = 0
    errors = 0
    
    pbar = tqdm(desc="Lignes", unit="l", position=1, leave=False)
    
    for line_num, line_bytes in enumerate(obj["Body"].iter_lines()):
        total_lines += 1
        pbar.update(1)
        
        if max_lines and total_lines > max_lines:
            break
        
        line = line_bytes.decode('utf-8', errors='ignore').strip()
        if not line:
            continue
        
        try:
            record = json.loads(line)
            valid_json += 1
            
            job_data = safe_get_job_data(record)
            if job_data and "name" in job_data:
                job_data_ok += 1
                
        except json.JSONDecodeError:
            errors += 1
    
    pbar.close()
    
    print(f"📊 {total_lines:,} lignes | {valid_json:,} JSON OK | {job_data_ok:,} job_data ({100*job_data_ok/total_lines:.2f}%)")
    return {"total": total_lines, "json_ok": valid_json, "job_data_ok": job_data_ok, "errors": errors}

# === CHOIX ===
TEST_MODE = "QUICK"  # "QUICK" ou "FULL"

if TEST_MODE == "QUICK":
    # 50 lignes/fichier → 5min
    for key in jsonld_keys[:10]:  # Top 10
        test_jsonl_loading(s3_boto, key, bucket, 50)
elif TEST_MODE == "FULL":
    # TOUS fichiers → 30min
    stats_all = []
    with tqdm(jsonld_keys, desc="FULL Test") as pbar:
        for key in pbar:
            stats = test_jsonl_loading_FULL(s3_boto, key, bucket)
            stats_all.append(stats)
    
    total = sum(s["total"] for s in stats_all)
    job_ok = sum(s["job_data_ok"] for s in stats_all)
    print(f"\n🎯 FINAL: total = {total} - job ok = {job_ok} soit {job_ok/total*100:.2f}% job_data OK")


# Lire WTTJ Json + Alimentation Class WTTJ 

In [ ]:
key = "welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000047.jsonl"

print(f"bucket={bucket}")
print(f"Key={key}")

print("- Get objec from S3 for key {key}")
obj = s3_boto.get_object(Bucket=bucket, Key=key)

print("- Read and decode json string")
lines = obj["Body"].read().decode("utf-8").splitlines()

print("- Deserialze json in dictionnary")
records = [json.loads(l) for l in lines if l.strip()]

print("- Dict. Keys = ", records[0].keys())
print("- Nb properties= " ,len(records))
print("")
#print(records[0].values())

wttj_all =[]
max_elmt=1000
print(f"- Parse {max_elmt}")

for i, record in enumerate(tqdm(records[:max_elmt], desc="Processing jobs")):

    #initial_data= get_json_field_from_record(records[i], "initial_data")    
    job_data = get_json_field_from_record(records[i], "job_data")

    wttj = WTTJ( 
        id = job_data["wttj_reference"],
        job_title = clean_html( job_data["name"] ),
        job_description = clean_html( job_data["description"] ),
        company_employees =0, # job_data["nb_employees"],
        profile = job_data["profile"],
        url = job_data["urls"][0]
    )
    wttj_all.append(wttj)

for a_wttj in wttj_all[:1]:
    print(a_wttj.job_title)


# Inférences ROME

In [ ]:
import json
import requests

for a_wttj in wttj_all[:10]:
    print(30* "=")
    payload = {
        "intitule": a_wttj.job_title,
        "description": a_wttj.job_description
    }
    print("title = " , a_wttj.job_title )
    print("description = ", a_wttj.job_description )

    r = requests.post("http://api:8000/predict", json=payload)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
    


# Debug

In [ ]:
# =============================================================================
# 📄 EXTRACTION window.__INITIAL_DATA__ depuis WelcomeToTheJungle
# Script Jupyter Notebook complet - Testé et fonctionnel
# =============================================================================

import re
import json
import requests
from typing import Optional, Dict, Any
from pathlib import Path
import json as json_module  # Alias pour éviter conflit

display("🚀 Initialisation...")

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

URL = "https://www.welcometothejungle.com/fr/companies/groupement-les-mousquetaires/jobs/caissiere-hote-de-caisse-h-f_carnoules"

# Regex OPTIMISÉE pour strings "..." avec échappements internes
INITIAL_DATA_RE = re.compile(
    r'window\.__INITIAL_DATA__\s*=\s*"((?:[^"\\]|\\.)*)"\s*;?',
    re.DOTALL | re.MULTILINE
)


display(f"📍 URL cible : {URL}")

# =============================================================================
# 2. FONCTION D'EXTRACTION
# =============================================================================

def extract_initial_data_from_html(html: str) -> Optional[Dict[str, Any]]:
    """Extrait et parse window.__INITIAL_DATA__ de l'HTML."""
    m = INITIAL_DATA_RE.search(html)
    if not m:
        print("❌ Pas de window.__INITIAL_DATA__ trouvé")
        return None
    
    raw_json = m.group(1)
    print(f"✅ Capturé {len(raw_json):,} caractères JSON")
    print(raw_json)
    
    try:
        data = json.loads(raw_json)
        print(f"✅ JSON parsé avec succès ! Clés : {list(data.keys())}")
        return data
    except json.JSONDecodeError as e:
        print(f"❌ Erreur JSON : {e}")
        return None

# =============================================================================
# 3. RÉCUPÉRATION ET EXTRACTION
# =============================================================================

print("📥 Téléchargement de la page...")
try:
    resp = requests.get(
        URL, 
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'},
        timeout=15
    )
    resp.raise_for_status()
except Exception as e:
    print(f"❌ Erreur requête : {e}")
    raise

html = resp.text
print(f"📄 HTML récupéré : {len(html):,} caractères")

# Sauvegarde HTML pour debug
Path("page_wttj.html").write_text(html, encoding="utf-8")
print("💾 HTML sauvé → page_wttj.html")

# Extraction !
data = extract_initial_data_from_html(html)

# =============================================================================
# 4. ANALYSE DES RÉSULTATS
# =============================================================================

if data:
    display("🎉 EXTRACTION RÉUSSIE !")
    
    # Sauvegarde JSON propre
    Path("initial_data_clean.json").write_text(
        json.dumps(data, indent=2, ensure_ascii=False), 
        encoding="utf-8"
    )
    print("💾 JSON propre sauvé → initial_data_clean.json")
    
    # Exploration des données
    if 'queries' in data and data['queries']:
        first_query = data['queries'][0]
        job_data = first_query.get('state', {}).get('data', {})
        
        display("## 📋 PREMIER JOB TROUVÉ")
        display(f"**Nom** : {job_data.get('name', 'N/A')}")
        display(f"**Ville** : {job_data.get('office', {}).get('city', 'N/A')}")
        display(f"**Slug** : {job_data.get('slug', 'N/A')}")
        display(f"**Champs obligatoires** : {len(job_data.get('application_fields', []))}")
        
        # Aperçu application_fields
        fields = job_data.get('application_fields', [])
        df_fields = pd.DataFrame(fields)[['name', 'mode']] if 'pd' in globals() else None
        
    else:
        display("ℹ️  Structure inattendue, mais JSON valide !")
        display(f"Clés racines : {list(data.keys())}")
        
else:
    print("🔍 DEBUG : Vérification présence variable...")
    if 'window.__INITIAL_DATA__' in html:
        print("✅ Variable présente dans HTML, problème regex")
    else:
        print("❌ Variable ABSENTE de cette page")

# =============================================================================
# 5. VALIDATION FINALE
# =============================================================================

print("\n✅" + "="*60)
print("RÉSUMÉ :")
print(f"• URL : {URL}")
print(f"• Taille JSON extrait : {len(json.dumps(data)) if data else 0:,} caractères")
print(f"• Fichiers générés : page_wttj.html, initial_data_clean.json")
print("="*60)


In [ ]:
import re
import json
from pathlib import Path
from typing import Dict, Any

def decode_wttj_data(html_content: str) -> Dict[str, Any]:
    """
    Extrait et parse window.__INITIAL_DATA__ de WTTJ (double serialisation)
    """
    # 1. Regex capture STRING COMPLET (échappements inclus)
    pattern = r'window\.__INITIAL_DATA__\s*=\s*"((?:[^"\\]|\\.)*)"\s*;?'
    match = re.search(pattern, html_content, re.DOTALL)
    
    if not match:
        raise ValueError("❌ window.__INITIAL_DATA__ non trouvé")
    
    raw_string = match.group(1)
    print(f"📏 Raw string: {len(raw_string):,} caractères")
    
    # 2. Décode Unicode JS (\uXXXX → UTF-8)
    json_text = bytes(raw_string, 'utf-8').decode('unicode_escape')
    
    # 3. Parse JSON principal
    data = json.loads(json_text)
    
    # 4. Dé-sérialise les champs internes (arrays/objects stringifiés)
    for key, value in data.items():
        if isinstance(value, str):
            try:
                parsed = json.loads(value)
                data[key] = parsed
                print(f"🔄 {key}: string → {type(parsed).__name__}")
            except json.JSONDecodeError:
                pass  # Garde string si pas JSON
    
    return data

# ═══════════════════════════════════════════════════════════════
# EXECUTION
# ═══════════════════════════════════════════════════════════════

html_file = Path("page_wttj.html")
if not html_file.exists():
    print("❌ page_wttj.html manquant")
else:
    html = html_file.read_text(encoding='utf-8')
    
    try:
        job_data = decode_wttj_data(html)
        
        # 📊 AFFICHAGE RÉSUMÉ
        print("\n🎉 EXTRACTION RÉUSSIE !")
        print("=" * 60)
        print(f"📂 Slug: {job_data.get('slug', 'N/A')}")
        print(f"📛 Titre: {job_data.get('name', 'N/A')}")
        print(f"🏢 Ville: {job_data.get('office', {}).get('city', 'N/A')}")
        print(f"📋 Contrat: {job_data.get('contractType', 'N/A')}")
        print(f"🔑 queryHash: {job_data.get('queryHash', 'N/A')}")
        
        # 💾 SAUVEGARDE
        output_file = Path("wttj_job_complete.json")
        output_file.write_text(
            json.dumps(job_data, indent=2, ensure_ascii=False),
            encoding='utf-8'
        )
        print(f"\n✅ SAUVEGARDÉ: {output_file.absolute()}")
        print(f"📏 Taille JSON: {len(json.dumps(job_data)):,} chars")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        print("\n🔍 DEBUG: premiers 300 chars raw:")
        match = re.search(r'window\.__INITIAL_DATA__\s*=\s*"(.{0,300})', html)
        if match:
            print(repr(match.group(1)))
